In [1]:
from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer
import json
import torch

from transformers_cfg.parser import parse_ebnf

from transformers_cfg.grammar_utils import IncrementalGrammarConstraint 
from transformers_cfg.generation.logits_process import GrammarConstrainedLogitsProcessor
from transformers_cfg.recognizer import StringRecognizer

# Detect if GPU is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# create new OrderedDict that does not contain `module.`
from collections import OrderedDict
def clean_and_load_CKPT(ckpt):
    
    checkpoint=torch.load(ckpt,map_location=torch.device('cpu'))
    new_state_dict = OrderedDict()
    for k, v in checkpoint["state_dict"].items():
        name = k.replace("model.model", "model") # remove `module.`
        if(k in ["model.final_logits_bias","model.lm_head.weight"]):
            name = k.replace("model.", "")
        new_state_dict[name] = v
    return new_state_dict


############# PATH OF FILES AND 
model_name="facebook/bart-base"



/home/cringwal/Desktop/Models_SAVE/env_shapes/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [5]:
MODELS=[
    {"model":"M-",
     "tkn_path":"./M-/DS_turtleS_0datatype_1inLine_1facto_BART_sample3_train_model_-_bart-base_tokenizer",
     "cpt_path":"./M-/checkpoints/last.ckpt"},
    {"model":"M0",
     "tkn_path":"./M0/DS_turtleS_0datatype_1inLine_1facto_BART_sample0_train_model_0_V2_bart-base_tokenizer",
     "cpt_path":"./M0/checkpoints/last.ckpt"},
    {"model":"M1+",
     "tkn_path":"./M1+/DS_turtleS_0datatype_1inLine_1facto_BART_sample0_train_model_1_V2_bart-base_tokenizer",
     "cpt_path":"./M1+/checkpoints/last.ckpt"},

]

In [22]:

examples = ["<s>Percy_Hynes_White : Percy Hynes White (born October 8, 2001) is a Canadian actor. He is known for his roles in films such as Edge of Winter and A Christmas Horror Story, for his role in the television series Between, and for his starring role as Andy Strucker in The Gifted.</s>",
           "<s>Evelyn_Sterling : Dr. Evelyn Sterling was born on July 12, 1978, in the quaint town of Oakridge, nestled in the hills of Vermont, USA. From a young age, she exhibited a keen interest in the stars, often spending nights gazing at the celestial wonders with her father's telescope. Her fascination with the cosmos grew as she devoured books on astronomy and physics.</s>"]


In [23]:
for MDL in MODELS:
##########################  LOAD MODEL AND TOKENIZER
    print("================",MDL["model"])
    tokenizer_kwargs = {
        "use_fast": True,
    #    "add_tokens": all_vocab
    }
    tokenizer = AutoTokenizer.from_pretrained(
                MDL["tkn_path"],
                **tokenizer_kwargs
            )



    config = AutoConfig.from_pretrained(
            model_name,
            decoder_start_token_id = 0,
            #early_stopping = False,
            no_repeat_ngram_size = 0,
            dropout=0.1,
            forced_bos_token_id=None,
        )

    model = AutoModelForSeq2SeqLM.from_config(
            config=config
        )

    model.resize_token_embeddings(len(tokenizer))

    ckpt=clean_and_load_CKPT(MDL["cpt_path"])

    model.load_state_dict( ckpt)


    ############################ TEST IT WITH 

    inputs = tokenizer(examples, return_tensors='pt', padding=True, truncation=True)
    gen_kwargs = {
                "max_length": 1024,
                "early_stopping": False,
                "length_penalty": 0,
                "no_repeat_ngram_size": 0,
                "num_beams": 2
            }

    translated_tokens = model.generate(inputs["input_ids"].to(model.device),
                attention_mask=inputs["attention_mask"].to(model.device),
                use_cache = True,
                **gen_kwargs)
    print( tokenizer.batch_decode(translated_tokens, skip_special_tokens=True, spaces_between_special_tokens = True))


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


================ M-


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[':Percy_Hynes_White a :Person; :label "Percy Hynes White"; :birthDate "2001-10-08"; :birthYear "2001".', ':Evelyn_Sterling a :Person; :label "Evelyn Sterling"; :birthDate "1978-07-12"; :birthYear "1978".']
================ M0


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[':Percy_Hynes_White a :Person; :label "Percy Hynes White"; :birthDate "2001-10-08"; :birthName "Percy Hynes White"; :birthYear "2001".', ':Evelyn_Sterling a :Person; :label "Evelyn Sterling".']
================ M1+


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[':Percy_Hynes_White a :Person; :label "Percy Hynes White"; :birthDate "2001-10-08"; :birthName "Percy Hynes White"; :birthYear "2001".', ':Evelyn_Sterling a :Person; :birthDate "1978-07-12"; :birthYear "1978".']
